In [ ]:
import pandas as pd

In [ ]:
# To access files in your Google Drive from Google Colaboratory, you need to authenticate your notebook with Google Drive. Run the following code to authenticate:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import PurePath
data_path = PurePath('/content/drive/My Drive/Colab Notebooks/AFL/AFL Fantasy Data')

In [ ]:
# read in the file - which in this case is in csv - comma separated value - format
processed_data_path = data_path / 'Processed'
filename = 'afl_fantasy_league_training_data-cleaned.csv'
afl_df = pd.read_csv(processed_data_path / filename)

In [ ]:
afl_df

In [ ]:
# First 20 data points
# Use context manager to temporarily allow all columns to be shown
with pd.option_context('display.max_columns', None):
    display(afl_df.head(20))

In [ ]:
with pd.option_context('display.max_columns', None):
    display(afl_df.describe())

In [ ]:
with pd.option_context('display.max_columns', None):
    display(afl_df.describe(include='all'))

In [ ]:
afl_df.info()

In [ ]:
# Generate dummies for categorical data
afl_df = pd.get_dummies(afl_df, columns=['Home Team','Away Team', 'Day', 'Weather Summary'], drop_first=True, dtype=int)

In [ ]:
_ = afl_df.plot(subplots=True, layout=(20,4), figsize=(20,100))

In [ ]:
# Scatter Plot Example
afl_df.plot.scatter(x='Home Total', y='Away Total')

In [ ]:
# Histogram Example
ax = afl_df.plot.bar(y='Temperature (°C)',x='Date', rot=90, )
_ = ax.set(xticklabels=[])

In [ ]:
x_train = afl_df.copy()
x_train.drop(
    columns=[
        'Home Goals', 'Home Points', 'Home Total',
        'Away Goals', 'Away Points', 'Away Total',
    ],
    inplace=True
)

# Date → Julian date
x_train['Date'] = pd.to_datetime(x_train['Date'], errors='coerce')
x_train['Date'] = x_train['Date'].apply(pd.Timestamp.to_julian_date)

# Time → minutes since midnight
x_train['Time'] = pd.to_datetime(x_train['Time'], errors='coerce')
x_train['Time'] = x_train['Time'].dt.hour * 60 + x_train['Time'].dt.minute

In [ ]:
# The outcome to predict is the score difference between teams
y_train = afl_df['Home Total'] - afl_df['Away Total']

In [ ]:
x_train.columns

In [ ]:
x_train.info()

In [ ]:
# We know we're going to get a dataset to predict the result (score difference) for
# Is it reasonable to fit an OLS to the current data, and use this same model to predict the test data?
# Let's simulate this by using a 10-fold CV process
# what does the array output mean? A: R^2 values for each fold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

ols = LinearRegression()
cross_val_score(ols, x_train, y_train, cv=10)


In [ ]:
# The predictions are pretty good (R^2 from predicted data > 0.99), and robust (in each CV run) - let's proceed with OLS
# We'll use statsmodel for this as it gives us a nicer output

import statsmodels.api as sm

In [ ]:
x_train = sm.add_constant(x_train)
model = sm.OLS(y_train.values,x_train).fit()

model.summary()

In [ ]:
# We'll now use this model to predict what the scores should be (for the training data), and look at the difference between the actual and predicted data
y_predict_train = model.predict(x_train)

In [ ]:
from sklearn.metrics import PredictionErrorDisplay, root_mean_squared_error, r2_score

In [ ]:
_ = PredictionErrorDisplay.from_predictions(y_true=y_train, y_pred=y_predict_train, kind="actual_vs_predicted").plot()

In [ ]:
display(root_mean_squared_error(y_true=y_train, y_pred=y_predict_train))
display(r2_score(y_true=y_train, y_pred=y_predict_train))

In [ ]:
# Now we get our hands on the data we need to predict
# You will need to generate a 'cleaned' dataset by running it through the same cleaning process as the training data
filename = 'afl_fantasy_league_test_data-cleaned.csv'

afl_test_df = pd.read_csv(processed_data_path / filename)
afl_test_df

In [ ]:
afl_test_df.isnull().sum()

In [ ]:
# Generate dummies for categorical data
afl_test_df = pd.get_dummies(afl_test_df, columns=['Home Team','Away Team', 'Day', 'Weather Summary'], drop_first=True, dtype=int)

x_test = afl_test_df.copy()

# Date → Julian date
x_test['Date'] = pd.to_datetime(x_test['Date'], errors='coerce')
x_test['Date'] = x_test['Date'].apply(pd.Timestamp.to_julian_date)

# Time processing: Convert 'Kickoff' column to minutes since midnight and assign to 'Time'
x_test['Time'] = pd.to_datetime(x_test['Time'], errors='coerce')
x_test['Time'] = x_test['Time'].dt.hour * 60 + x_test['Time'].dt.minute

In [ ]:
# Do our columns match?
display(len(x_train.columns))
display(len(x_test.columns))

In [ ]:
#test and train datasets have the same number of columns for the explanatory variables BUT the explanatory variables have fewer values in them for the test set and thus when creating binary variables, we miss out on some of the categories
display(set(x_train.columns) - set(x_test.columns))
display(set(x_test.columns) - set(x_train.columns))

In [ ]:
# We need to redo the dummy generation process, this time including the missing dummies so that all variables are available for comparison with the training data when reindexing
afl_test_df = pd.read_csv(processed_data_path / filename)

# Generate dummies for categorical data
afl_test_df = pd.get_dummies(afl_test_df, columns=['Home Team','Away Team', 'Day', 'Weather Summary'], drop_first=False, dtype=int)

x_test = afl_test_df.copy()

# Date → Julian date
x_test['Date'] = pd.to_datetime(x_test['Date'], errors='coerce')
x_test['Date'] = x_test['Date'].apply(pd.Timestamp.to_julian_date)

# Time processing: Convert 'Kickoff' column to minutes since midnight and assign to 'Time'
x_test['Time'] = pd.to_datetime(x_test['Time'], errors='coerce')
x_test['Time'] = x_test['Time'].dt.hour * 60 + x_test['Time'].dt.minute

# And reindex the columns based on the training data
# Re-indexing means matching the columns from x_train to x_test
# put columns from train set into test set: e.g. weather - "rain" does not exist in the test (EXAMPLE) then fills this column for the test
# Note we are re-indexing (inserting columns) for only those for x_train df (not the y_train)
x_test = x_test.reindex(labels=x_train.columns, axis='columns', fill_value=0)
x_test['const'] = 1

with pd.option_context('display.max_columns', None):
    display(x_test)

In [ ]:
display(len(x_train.columns))
display(len(x_test.columns))

In [ ]:
# Let's make our prediction!
# we are predicting the 'home goals' - 'away goals' = point difference
# the "model" was trained on the train data set - cleaned version...and now we bring this model over to the test set to predict the outcome
y_predict_test = model.predict(x_test)
y_predict_test

In [ ]:
# Now we've got the actual data - let's look at how good our prediction was

filename = 'afl_fantasy_league_test_data_with_result.csv'
afl_result_df = pd.read_csv(data_path / filename)

In [ ]:
afl_result_df['Point Difference']

In [ ]:
display(root_mean_squared_error(y_true=afl_result_df['Point Difference'], y_pred=y_predict_test))
display(r2_score(y_true=afl_result_df['Point Difference'], y_pred=y_predict_test))

In [ ]:
_ = PredictionErrorDisplay.from_predictions(y_true=afl_result_df['Point Difference'], y_pred=y_predict_test, kind="actual_vs_predicted").plot()

In [ ]:
# The predictions we've got are nowhere near as good as what we got with the training data - what's going on?

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x=y_predict_train, y=x_train['Random Column'])
plt.scatter(x=afl_result_df['Point Difference'], y=x_test['Random Column'])
plt.show()

In [ ]:
# Looks like the 'Random Column' isn't actually so random in the train set (blue dots). But it is random in the test set (orange dots).
# Do we get a better prediction if we leave this out?